# 03 · Async labelling pipeline

Scoring ~57,000 posts one request at a time was too slow, so this version runs requests concurrently with `asyncio` + `aiohttp`:
- a semaphore caps concurrency (10 requests at a time) to stay within API limits,
- failed requests are retried,
- results are appended to disk as they arrive, and IDs already in the output file are skipped on restart (resume after interruption).

This first full pass also tagged each post's role (Community / Speculation / Celebrity) and produced `data/Doge_Sentiment_Master.csv`. The final scores used in the paper come from the ATF prompt in `scripts/score_emotions_atf.py`.

Set `DEEPSEEK_API_KEY` in your environment before running.

In [ ]:
!pip install aiohttp tqdm nest_asyncio

In [ ]:
import pandas as pd
import json
import asyncio
import aiohttp
import os
import nest_asyncio
from tqdm.asyncio import tqdm

nest_asyncio.apply()

# --- 配置 ---
API_KEY = os.environ["DEEPSEEK_API_KEY"]
URL = "https://api.deepseek.com/v1/chat/completions"
INPUT_FILE = "todo_part_3.csv" #可以替换成file path
OUTPUT_FILE = "results_part_3_fast.csv"
CONCURRENCY = 10  

# --- 核心函数 ---
async def analyze_row(session, row, sem):
    async with sem:
        prompt = f"""
        Analyze Dogecoin sentiment for: "{row['body']}"
        Categorize: [Celebrity, Speculation, Community]
        Score (0-1): Joy, Trust, Fear, Surprise, Sadness, Anger.
        Return JSON ONLY: {{"cat": "...", "joy": 0.0, "tru": 0.0, "fea": 0.0, "sur": 0.0, "sad": 0.0, "ang": 0.0}}
        """
        
        payload = {
            "model": "deepseek-chat",
            "messages": [{"role": "user", "content": prompt}],
            "response_format": {"type": "json_object"}
        }
        
        for _ in range(3): 
            try:
                async with session.post(URL, json=payload, headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30) as resp:
                    data = await resp.json()
                    res = json.loads(data['choices'][0]['message']['content'])
                    
                    res['id'] = row['id']
                    res['date'] = row['date']
                    res['time'] = row['time']
                    return res
            except:
                await asyncio.sleep(1)
        return None

async def main():
    # 1. 加载数据
    df = pd.read_csv(INPUT_FILE)
    
    # 如果已经跑了一部分，自动跳过（断点续传）
    if os.path.exists(OUTPUT_FILE):
        done_df = pd.read_csv(OUTPUT_FILE)
        done_ids = set(done_df['id'].unique())
        df = df[~df['id'].isin(done_ids)]
        print(f"检测到进度，跳过已完成的 {len(done_ids)} 条。")

    tasks = []
    sem = asyncio.Semaphore(CONCURRENCY)
    
    async with aiohttp.ClientSession() as session:
        # 建立所有任务
        for _, row in df.iterrows():
            tasks.append(analyze_row(session, row, sem))
        
        # 使用进度条运行
        print(f"🚀 开始高速处理 {len(tasks)} 条数据...")
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            result = await f
            if result:
                # 实时写入文件
                temp_df = pd.DataFrame([result])
                header = not os.path.exists(OUTPUT_FILE)
                temp_df.to_csv(OUTPUT_FILE, mode='a', index=False, header=header)

# 执行
await main()